In [ ]:
# imports
import ultralytics
ultralytics.checks()
from ultralytics import YOLO


import sys
import os
import shutil
from dotenv import load_dotenv
load_dotenv()

In [22]:
sys.path.append(os.path.abspath('..'))

from src.data import download
from src.data.augment import process_dataset
from src.utils import plot_metrics

In [15]:
RAW_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"

# Data Preparation

## Download

In [24]:
download.download_dataset(RAW_PATH)

loading Roboflow workspace...
loading Roboflow project...
Raw data downloaded to: /home/maro/final-projects/Bowl_Score/cv-pipeline/data/raw


## Augment

In [23]:
print("Augmenting Training Set...")
process_dataset(
    raw_img_dir=RAW_PATH+"/train/images",
    raw_label_dir=RAW_PATH+"/train/labels",
    output_img_dir=PROCESSED_PATH+"/train/images",
    output_label_dir=PROCESSED_PATH+"/train/labels"
)

print("Augmenting Validation Set...")
process_dataset(
    raw_img_dir=RAW_PATH+"/valid/images",
    raw_label_dir=RAW_PATH+"/valid/labels",
    output_img_dir=PROCESSED_PATH+"/valid/images",
    output_label_dir=PROCESSED_PATH+"/valid/labels"
)

yaml_src = RAW_PATH+"/data.yaml"
yaml_dst = PROCESSED_PATH+"/data.yaml"

if os.path.exists(yaml_src):
    with open(yaml_src, "r") as f:
        yaml_content = f.read()

    # Update paths to point to the new processed directory
    yaml_content = yaml_content.replace(RAW_PATH, PROCESSED_PATH)

    with open(yaml_dst, "w") as f:
        f.write(yaml_content)

print("Augmentation Complete! Data is ready for YOLOv8 fine-tuning.")

Augmenting Training Set...
Skipping venue_2026-04-13-12-37-45_f000750_jpg.rf.70e941df96f61e01c4cc0bba1de97f2e.jpg due to bounding box clipping: Expected x_max for bbox [    0.35078     0.21484      1.0711     0.67266           2] to be in the range [0.0, 1.0], got 1.0710937976837158.
Augmenting Validation Set...
Augmentation Complete! Data is ready for YOLOv8 fine-tuning.


# Model fine-tuning

In [25]:
model = YOLO('yolov8n.pt')

In [26]:
results = model.train(
    data='../data/processed/data.yaml',
    epochs=100,             # Start with 100, YOLO has early stopping if it peaks early
    imgsz=640,              # The native resolution for YOLOv8
    batch=8,               # Standard batch size, lower to 8 if you hit GPU memory limits
    freeze=10,              # THE PRO-MOVE: Freezes the first 10 layers (the backbone)
    project='bowling_tracker',
    name='v1_frozen_backbone',
    device=0,               # Set to 'cpu' if you don't have a local GPU
    optimizer='auto',
    lr0=0.01                # Standard initial learning rate for transfer learning
)

Ultralytics 8.4.46 🚀 Python-3.13.3 torch-2.11.0+cu130 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 1
os.environ['CUDA_VISIBLE_DEVICES']: None


In [ ]:
# Cell 5: Validate the Model
metrics = model.val()
print(f"Mean Average Precision (mAP50-95): {metrics.box.map}")

In [ ]:
plot_metrics()